In [5]:
import tensorflow as tf
import numpy as np
import pandas as pd

input_size = 200
num_sensors = 4
num_classes = 3

def load_and_split_raw(csv_file, test_size=0.2):
    print(f"Loading {csv_file}...")
    df = pd.read_csv(csv_file)
    
    val_cols = [f'val_{i}' for i in range(60)]
    raw_packets = df[val_cols].values
    
    data = raw_packets.reshape(-1, 4) 
    
    split_idx = int(len(data) * (1 - test_size))
    train_data = data[:split_idx]
    test_data = data[split_idx:]
    return train_data, test_data

raw_train_relax, raw_test_relax = load_and_split_raw('../relax.csv')
raw_train_snap,  raw_test_snap  = load_and_split_raw('../fingersnap.csv')
raw_train_fist,  raw_test_fist  = load_and_split_raw('../fist.csv')

def make_windows(raw_data, label, window_size=input_size, step=50):
    windows = []
    labels = []
    for i in range(0, len(raw_data) - window_size + 1, step):
        windows.append(raw_data[i : i + window_size])
        labels.append(label)
    return np.array(windows), np.array(labels)

X_train_relax, y_train_relax = make_windows(raw_train_relax, 0)
X_test_relax,  y_test_relax  = make_windows(raw_test_relax, 0)
X_train_snap,  y_train_snap  = make_windows(raw_train_snap, 1)
X_test_snap,   y_test_snap   = make_windows(raw_test_snap, 1)
X_train_fist,  y_train_fist  = make_windows(raw_train_fist, 2)
X_test_fist,   y_test_fist   = make_windows(raw_test_fist, 2)

X_train = np.vstack((X_train_relax, X_train_snap, X_train_fist)).astype(np.float32)
y_train = np.concatenate((y_train_relax, y_train_snap, y_train_fist))
X_test = np.vstack((X_test_relax, X_test_snap, X_test_fist)).astype(np.float32)
y_test = np.concatenate((y_test_relax, y_test_snap, y_test_fist))

shuffle_indices = np.random.permutation(len(X_train))
X_train = X_train[shuffle_indices]
y_train = y_train[shuffle_indices]

tf.keras.backend.clear_session()

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(input_size, num_sensors)),
    tf.keras.layers.Conv1D(40, 10, strides=4, padding='same', activation='relu'),
    tf.keras.layers.Conv1D(25, 5, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(100, 5, padding='same', activation='relu'),
    tf.keras.layers.Conv1D(50, 5, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Conv1D(100, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.summary()

# Learn
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_emg_model.keras", save_best_only=True, monitor="val_accuracy"
)
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32, callbacks=[checkpoint]
)

best_model = tf.keras.models.load_model("best_emg_model.keras")

# Quantization
def representative_data_gen():
    indices = np.random.choice(len(X_train), 100, replace=False)
    for i in indices:
        yield [X_train[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

with open("emg_model.tflite", "wb") as f:
    f.write(tflite_model)
print("Model emg_model.tflite saved!")

Loading ../relax.csv...
Loading ../fingersnap.csv...
Loading ../fist.csv...


/home/dariy/emg-lab/source/python_scripts/emg_venv/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 50, 40)         │         1,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 25)         │         5,025 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 25, 25)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 25, 100)        │        12,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 25, 50)         │        25,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 12, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 12, 100)        │        15,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 100)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,718 (233.27 KB)

...

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8131 - loss: 1.1266 - val_accuracy: 0.8638 - val_loss: 0.4913
Epoch 2/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8857 - loss: 0.5372 - val_accuracy: 0.8388 - val_loss: 0.5008
Epoch 3/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9077 - loss: 0.3922 - val_accuracy: 0.8735 - val_loss: 0.4520
Epoch 4/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9107 - loss: 0.3148 - val_accuracy: 0.8783 - val_loss: 0.4330
Epoch 5/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9165 - loss: 0.2834 - val_accuracy: 0.8928 - val_loss: 0.3615
Epoch 6/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9153 - loss: 0.3018 - val_accuracy: 0.8928 - val_loss: 0.3718
Epoch 7/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9187 - loss: 0.2555 - val_accuracy: 0.8751 - val_loss: 0.4679
Epoch 8/30
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9275 - loss: 0.2232 - val_accuracy: 0.

INFO:tensorflow:Assets written to: /tmp/tmptv03ot86/assets


Saved artifact at '/tmp/tmptv03ot86'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 4), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139840448853968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448848592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448840528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448854160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448847632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448846288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448853008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448848208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448842832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448842064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139840448842640: Ten

/home/dariy/emg-lab/source/python_scripts/emg_venv/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1783637575.244200  539391 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1783637575.244209  539391 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1783637575.244295  539391 reader.cc:83] Reading SavedModel from: /tmp/tmptv03ot86
I0000 00:00:1783637575.244719  539391 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1783637575.244727  539391 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmptv03ot86
I0000 00:00:1783637575.248839  539391 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1783637575.268440  539391 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmptv03ot86
I0000 00:00:1783637575.274978  539391 loader.cc:471] S